# Bisaya HMM-GMM SAT Training (Kaldi) -- Kaggle Edition

A Kaggle-native rewrite of `train.ipynb`. Same single pipeline -- HMM-GMM
with speaker-adaptive training (SAT), a 2-gram language model, and the PS27
phoneme set, per `conclusion.md`'s "Conclusions and Recommendations" (best
Bisaya model overall: 2-gram, 3-state, SAT HMM-GMM using PS27, 5.41% WER,
beating every DNN/TDNN variant tested). See `train.ipynb`'s Section 1 for
the full reasoning behind that choice -- it isn't repeated here.

This notebook differs from `train.ipynb` in two ways:

1. **Paths are resolved directly against this project's actual Kaggle input
   dataset** (`troymerales/bisaya-audio`) instead of a generic placeholder.
2. **Every expensive step checks for its own completed output first and
   skips straight past it if found** -- not just the GMM stages (which
   `train.ipynb` already made resumable). Kaldi's build, data prep, feature
   extraction, lexicon/LM/graph building, and decoding all do this too now.
   Re-running this notebook top to bottom after a Kaggle session restart
   should cost almost nothing except whichever single step didn't finish.

**Why data loading looks different from `kagglebasis.ipynb`:** that
notebook pre-built combined `train.parquet`/`test.parquet` files by
concatenating all shards with `pa.concat_tables()` and writing them out with
`pq.write_table()`, then read them back with `pd.read_parquet()`. That
round-trip hit a real gap in this Kaggle image's pyarrow: reading back a
multi-row-group file with a nested (`audio` struct) column raises
`ArrowNotImplementedError: Nested data conversions not implemented for
chunked array outputs`. Forcing everything into a single row group instead
(`combine_chunks()`) just trades that for `ArrowInvalid: offset overflow
while concatenating arrays`, since the combined audio bytes exceed the 2 GB
cap on Arrow's default 32-bit binary offsets. Loading each shard straight
into pandas with `pd.read_parquet()` and concatenating in memory -- exactly
what `train.ipynb` and `main.ipynb` already do successfully, with no
intermediate Parquet rewrite -- sidesteps both failure modes entirely, so
that's what Section 7 below does.

**Environment requirement:** Kaldi does not build on native Windows. This
notebook must run in a Linux environment -- a Kaggle notebook, or WSL2 on
your own machine.

## 1. Decisions Made Where the Paper/Guide Leaves Things Unspecified

Unchanged from `train.ipynb` -- see that notebook's Section 1 for the full
table and reasoning (model architecture, phoneme set, n-gram order, HMM
topology, CMVN scope, speed perturbation, LM weight sweep, split seed all
carry over identically). The only things new in this notebook are where
paths point and the resumability wrapping described above.

## 2. Configuration and Kaggle Checkpoint Persistence

`CORPUS_DIR` points directly at this project's Kaggle input dataset. The
corpus is **not** pre-split into train/test shards -- Section 7/8 load every
shard and derive a speaker-independent split in memory, so there is no
separate "split-generation" step and nothing gets written back out as a
recombined Parquet file.

`WORK_DIR` holds everything this notebook creates (`kaldi/`, `data/`,
`mfcc/`, `exp/`). `save_checkpoint()`/`restore_checkpoint()` tar/untar the
whole thing to survive a killed Kaggle session -- see `train.ipynb` Section 2
for the full commit/restore workflow across sessions.

In [ ]:
import os
import subprocess
import tarfile
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()

# Same path kagglebasis.ipynb used. If you already built Kaldi there in the
# current Kaggle session, pointing at the same WORK_DIR means Section 4/5
# below will detect it and skip rebuilding -- no need to touch this unless
# you want a clean separate working directory.
WORK_DIR = Path("/kaggle/working/bisaya_asr") if IS_KAGGLE else Path("./asr_train").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

KALDI_ROOT = WORK_DIR / "kaldi"
DATA_ROOT = WORK_DIR / "data"
MFCC_ROOT = WORK_DIR / "mfcc"
EXP_ROOT = WORK_DIR / "exp"

# Resolved directly against this project's actual Kaggle input dataset --
# not a generic placeholder. Not pre-split into train/test; Section 7/8
# below load every shard and derive a speaker-independent split in memory.
CORPUS_DIR = (
    Path("/kaggle/input/datasets/troymerales/bisaya-audio")
    if IS_KAGGLE
    else Path("data/bisaya_audio").resolve()
)

CHECKPOINT_PATH = (
    Path("/kaggle/working/checkpoint.tar.gz")
    if IS_KAGGLE
    else WORK_DIR.parent / "checkpoint.tar.gz"
)

print(f"IS_KAGGLE   = {IS_KAGGLE}")
print(f"WORK_DIR    = {WORK_DIR}")
print(f"CORPUS_DIR  = {CORPUS_DIR}")
if KALDI_ROOT.exists():
    print(f"Found existing Kaldi checkout at {KALDI_ROOT} -- build stages below will skip if already compiled.")

In [ ]:
import shutil

def sh(cmd, cwd=None, check=True, env=None):
    # Streams output live -- used for every Kaldi binary/script invocation
    # below instead of `!` magics, so it behaves the same interactively or
    # when a committed Kaggle version replays the notebook top to bottom.
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, cwd=cwd, check=check, executable="/bin/bash", env=env)
    return proc.returncode


def save_checkpoint():
    # Shells out to `tar` (with `pigz` for parallel gzip if installed)
    # instead of Python's tarfile module. Two reasons: (1) tarfile.add() on
    # a multi-GB Kaldi checkout gives zero output until it's completely
    # done -- from the notebook it looks identical to a hang for however
    # long compression takes; `tar -v` here prints each file as it's
    # archived, so progress is visible. (2) tarfile's gzip is
    # single-threaded; pigz uses all cores, which matters once src/ has a
    # full set of compiled binaries. `.git` history and leftover .o object
    # files are excluded -- neither is needed to resume training.
    # `set -o pipefail` matters here: without it, a failing `tar` piped
    # into `tail` would still report success (tail's own exit code).
    print(f"Archiving {WORK_DIR} -> {CHECKPOINT_PATH} ...")
    compressor = "pigz" if shutil.which("pigz") else "gzip"
    sh(
        f"set -o pipefail && tar -cv --use-compress-program={compressor} "
        f"--exclude='*/.git' --exclude='*.o' "
        f"-f {CHECKPOINT_PATH} -C {WORK_DIR.parent} {WORK_DIR.name} "
        f"| tail -n 5",
    )
    size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
    print(f"Done. {size_mb:.1f} MB.")


def restore_checkpoint(archive_path=None):
    archive_path = Path(archive_path) if archive_path else CHECKPOINT_PATH
    print(f"Restoring {archive_path} -> {WORK_DIR.parent} ...")
    decompressor = "pigz" if shutil.which("pigz") else "gzip"
    sh(f"set -o pipefail && tar -xv --use-compress-program={decompressor} "
       f"-f {archive_path} -C {WORK_DIR.parent} | tail -n 5")
    print("Done. Re-run the cells below -- every stage checks for its own "
          "completed output first and skips straight past it if found.")


def stage(name, done, fn, checkpoint=True):
    # Generic resumability helper used by every expensive step in this
    # notebook. `done` is a zero-arg callable returning True if this
    # stage's output already exists; `fn` does the actual work. Keeps
    # re-running this notebook top-to-bottom after a Kaggle session
    # restart cheap -- only whatever didn't finish last time actually runs.
    if done():
        print(f"[skip] {name}: output already exists")
        return
    print(f"[run]  {name}")
    fn()
    if checkpoint:
        save_checkpoint()


# --- Optional: push checkpoints to a personal Kaggle Dataset mid-session ---
# Only needed if mid-session crashes (not just session-end timeouts) are a
# real concern; requires the Kaggle API configured (kaggle.json).
#
# KAGGLE_DATASET_SLUG = "your-username/asr-train-checkpoint"
# def push_checkpoint_to_dataset():
#     save_checkpoint()
#     sh(f"kaggle datasets version -p {CHECKPOINT_PATH.parent} -m 'checkpoint update' "
#        f"-d {KAGGLE_DATASET_SLUG}")

## 3. Install Dependencies

Skipped entirely on a re-run once `.deps_installed` exists under
`WORK_DIR` -- `apt-get`/`pip` are themselves idempotent, but skipping the
call outright avoids the overhead of `apt-get update` on every re-run.

In [ ]:
import shutil

DEPS_MARKER = WORK_DIR / ".deps_installed"

def _install_deps():
    sh("apt-get update -qq && apt-get install -y -qq "
       "build-essential automake autoconf libtool subversion git zlib1g-dev "
       "gfortran libatlas-base-dev sox perl cmake libboost-all-dev libeigen3-dev")
    sh("pip install -q tqdm")
    DEPS_MARKER.touch()

stage("apt/pip dependencies", DEPS_MARKER.exists, _install_deps, checkpoint=False)

# The checks below are separate from the marker above (which may already
# exist from an earlier run that predates one of these being added to the
# list) -- each is checked and installed independently so a resumed
# session still gets it even if the main install was skipped entirely.
#
# perl: Kaldi's steps/utils scripts are Perl; missing it fails with a bare
# "exit status 127" wherever the first .pl script runs.
stage(
    "install perl (if missing)",
    lambda: shutil.which("perl") is not None,
    lambda: sh("apt-get install -y -qq perl"),
    checkpoint=False,
)

# cmake: checked by absolute apt install path (/usr/bin/cmake), NOT
# shutil.which("cmake") -- this Kaggle image ships a pip-installed cmake
# that resolves first on PATH (pip's console-script shim lands in
# /usr/local/bin, which precedes /usr/bin), so shutil.which would find
# that one and consider this stage already done. Pinning to apt's cmake
# via absolute path (Section 4 does the same) just keeps the toolchain
# deterministic; it turned out NOT to be the actual fix for the Boost
# detection issue there (see Section 4's comment) -- that's a genuine
# Debian/Ubuntu Boost packaging gap, independent of cmake version.
stage(
    "install cmake (if missing)",
    lambda: Path("/usr/bin/cmake").exists(),
    lambda: sh("apt-get install -y -qq cmake"),
    checkpoint=False,
)

# boost: KenLM's own CMake build (Section 4, our direct build -- not
# Kaldi's restricted install_kenlm_query_only.sh) needs Boost to produce
# lmplz. libboost-all-dev is heavier than a targeted subset, but avoids
# yet another single-missing-component 127 after everything else built.
stage(
    "install boost + eigen (if missing)",
    lambda: Path("/usr/include/boost/version.hpp").exists(),
    lambda: sh("apt-get install -y -qq libboost-all-dev libeigen3-dev"),
    checkpoint=False,
)

## 4. Kaldi Recipe: Clone + Build Tools + KenLM

Clone is skipped if `KALDI_ROOT` already exists (e.g. restored from a
checkpoint). The `tools/` build (OpenFST etc.) and KenLM install are each
skipped once their own output binary/library is found, rather than
re-invoking `make`/the installer script every time.

In [ ]:
def _kaldi_clone_complete():
    # KALDI_ROOT.exists() alone is NOT reliable: an interrupted clone
    # (session killed/timed out mid-clone) still leaves the directory on
    # disk, just with an incomplete tree -- and this stage would then skip
    # re-cloning forever, silently. Check a few files that only exist after
    # a genuinely complete checkout instead, spread across the tree (not
    # just near the root, since a partial clone can still get early files).
    canaries = [
        KALDI_ROOT / "tools" / "extras" / "install_kenlm_query_only.sh",
        KALDI_ROOT / "egs" / "wsj" / "s5" / "steps",
        KALDI_ROOT / "egs" / "wsj" / "s5" / "utils",
        KALDI_ROOT / "src" / "Makefile",
    ]
    return KALDI_ROOT.exists() and all(c.exists() for c in canaries)


def _clone_kaldi():
    if KALDI_ROOT.exists():
        # Repair, don't blow away: KALDI_ROOT exists but failed the
        # completeness check above, which most likely means tools/ (e.g.
        # OpenFST) was already built on top of this same incomplete tree.
        # Re-cloning into a temp dir and only copying in files that are
        # currently MISSING (cp -n = no-clobber) fills the gaps without
        # touching -- or wasting -- anything already built.
        print(f"{KALDI_ROOT} exists but is missing files (incomplete clone) -- repairing in place.")
        repair_dir = KALDI_ROOT.parent / "kaldi_repair_tmp"
        sh(f"rm -rf {repair_dir}")
        sh(f"git clone --depth 1 https://github.com/kaldi-asr/kaldi.git {repair_dir}")
        sh(f"cp -rn {repair_dir}/. {KALDI_ROOT}/")
        sh(f"rm -rf {repair_dir}")
    else:
        sh(f"git clone --depth 1 https://github.com/kaldi-asr/kaldi.git {KALDI_ROOT}")

stage("clone Kaldi", _kaldi_clone_complete, _clone_kaldi)


def _build_tools():
    sh("make -j$(nproc)", cwd=KALDI_ROOT / "tools")

stage(
    "build Kaldi tools (OpenFST etc.)",
    lambda: (KALDI_ROOT / "tools" / "openfst" / "lib").exists(),
    _build_tools,
)


def _install_kenlm():
    # Kaldi's own tools/extras/install_kenlm_query_only.sh deliberately
    # does NOT build lmplz -- straight from that script's header: "this
    # script doesn't install a full-build of kenlm... a full-build (with
    # arpa counting/smoothing/interpolation support) depends on EIGEN and
    # BOOST, too heavy to integrate." Section 11 needs lmplz to build the
    # 2-gram ARPA, so we build KenLM ourselves instead: Boost/Eigen already
    # installed in Section 3, clone kenlm directly, build it with its own
    # CMake, unmodified.
    #
    # -DBoost_NO_BOOST_CMAKE=ON is the actual fix for the
    # "Could not find a package configuration file... boost_program_options"
    # error, confirmed against BOTH cmake 3.31 (pip) and cmake 3.22.1
    # (apt) -- it's not a cmake-version issue at all. CMake's FindBoost
    # module tries Boost's own installed CMake package (BoostConfig.cmake)
    # first, before any legacy header/library search. Ubuntu's
    # libboost-all-dev ships that umbrella config file but not the
    # per-component ones it then needs (boost_program_optionsConfig.cmake
    # etc.) -- a known Debian/Ubuntu packaging gap -- so CMake commits to
    # the config-mode path (since the umbrella file IS found) and only
    # fails deeper in, instead of ever falling back. Boost_NO_BOOST_CMAKE=ON
    # skips Boost's CMake package entirely and forces genuine
    # library/header search, which works fine since libboost-all-dev does
    # provide the actual .so files and headers, just not those specific
    # per-component CMake config files.
    kenlm_dir = KALDI_ROOT / "tools" / "kenlm"
    sh(f"rm -rf {kenlm_dir}")
    sh(f"git clone https://github.com/kpu/kenlm.git {kenlm_dir}")
    sh("mkdir -p build && cd build && "
       "/usr/bin/cmake -DBoost_NO_BOOST_CMAKE=ON .. && "
       "make -j$(nproc)", cwd=kenlm_dir)

stage(
    "install KenLM (full build, for lmplz)",
    lambda: (KALDI_ROOT / "tools" / "kenlm" / "build" / "bin" / "lmplz").exists(),
    _install_kenlm,
)

## 5. Kaldi Recipe: Build Kaldi Binaries (`src/`)

Skipped entirely if `compute-mfcc-feats` already exists -- this is the
slowest step in the whole notebook (30-90 minutes cold), so avoiding a
redundant `make` invocation (even an incremental no-op one) matters most
here.

In [ ]:
src_dir = KALDI_ROOT / "src"
mfcc_bin = src_dir / "bin" / "compute-mfcc-feats"

# use-cuda=no: targets CPU-only environments (no CUDA-capable GPU assumed).
# If your Kaggle session has a GPU attached, drop --use-cuda=no.
def _build_kaldi_src():
    if not (src_dir / "kaldi.mk").exists():
        sh("./configure --shared --use-cuda=no", cwd=src_dir)
    sh("make -j$(nproc) depend", cwd=src_dir)
    sh("make -j$(nproc)", cwd=src_dir)

stage("build Kaldi (src/)", mfcc_bin.exists, _build_kaldi_src)

print("Kaldi build complete." if mfcc_bin.exists()
      else "WARNING: expected binary not found -- check the build log above for errors.")

## 6. Kaldi Recipe Scaffolding (`path.sh`, `cmd.sh`, `steps/`, `utils/`)

Writing `path.sh`/`cmd.sh` and symlinking `steps/`/`utils/` is already cheap
and idempotent (the symlink check below skips re-linking), so this isn't
wrapped in `stage()`.

In [ ]:
path_sh_lines = [
    f"export KALDI_ROOT={KALDI_ROOT}",
    "export PATH=$PWD/utils/:$KALDI_ROOT/tools/openfst/bin:$PWD:$PATH",
    ". $KALDI_ROOT/tools/config/common_path.sh",
    "export LC_ALL=C",
]
(WORK_DIR / "path.sh").write_text("\n".join(path_sh_lines) + "\n")

cmd_sh_lines = [
    "export train_cmd=run.pl",
    "export decode_cmd=run.pl",
    "export mkgraph_cmd=run.pl",
]
(WORK_DIR / "cmd.sh").write_text("\n".join(cmd_sh_lines) + "\n")

wsj_s5 = KALDI_ROOT / "egs" / "wsj" / "s5"
for name in ("steps", "utils"):
    link = WORK_DIR / name
    if not link.exists():
        link.symlink_to(wsj_s5 / name)

# Sanity check: a broken symlink (e.g. Kaldi's src/ build didn't finish, or
# WORK_DIR was restored from a checkpoint pointing at a different KALDI_ROOT)
# makes every steps/utils/*.pl call fail later with a bare, confusing
# "exit status 127" ("command not found") -- catching it here instead gives
# an immediate, specific error.
probe = WORK_DIR / "utils" / "utt2spk_to_spk2utt.pl"
assert probe.exists(), (
    f"{probe} not found -- steps/utils symlinks are broken (check that "
    f"{wsj_s5} exists and KALDI_ROOT={KALDI_ROOT} matches the Kaldi checkout "
    f"these were linked from)."
)

print("path.sh, cmd.sh, steps/, utils/ ready under", WORK_DIR)

## 7. Data Loading (Kaggle Corpus, No Intermediate Parquet Rewrite)

Reads every shard in `CORPUS_DIR` straight into pandas with
`pd.read_parquet()` and concatenates in memory -- no `pa.concat_tables()` +
`pq.write_table()` round trip (see this notebook's intro cell for why that
combination fails on this Kaggle image's pyarrow for a nested `audio`
column). Each shard is read exactly once, exactly the way it already
exists on disk.

In [ ]:
import pandas as pd

def load_all_shards(corpus_dir):
    files = sorted(Path(corpus_dir).glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    dfs = []
    for file in files:
        print(f"Loading: {file.name}")
        temp = pd.read_parquet(file)
        temp["source_file"] = file.name
        dfs.append(temp)
    return pd.concat(dfs, ignore_index=True)


corpus_df = load_all_shards(CORPUS_DIR)
print(f"\nLoaded {len(corpus_df):,} utterances from {CORPUS_DIR}")
print(f"Speakers: {corpus_df['speaker_id'].nunique() if len(corpus_df) else 0:,}")

## 8. Speaker-Independent Train/Test Split + Kaldi Data Dirs

Split **by speaker**, not by utterance, ~80/20, with a fixed seed (printed
below) so it's reproducible on rerun -- matches `train.ipynb` Section 6.
`build_kaldi_data_dir()` (which writes every utterance's audio out as an
individual `.wav` file plus `wav.scp`/`text`/`utt2spk`) is skipped per split
if that split's `wav.scp` already exists, since re-writing thousands of wav
files on every re-run is exactly the kind of redundant work this notebook
is meant to avoid.

In [ ]:
import re

SPLIT_SEED = 42  # fixed and printed so this split is reproducible on rerun

def speaker_independent_split(df, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(df["speaker_id"].unique())
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])

    assert train_speakers.isdisjoint(test_speakers)

    return df[df["speaker_id"].isin(train_speakers)].copy(), df[df["speaker_id"].isin(test_speakers)].copy()


train_df, test_df = speaker_independent_split(corpus_df)

print(f"seed = {SPLIT_SEED}")
print(f"train: {train_df['speaker_id'].nunique()} speakers, {len(train_df)} utterances")
print(f"test:  {test_df['speaker_id'].nunique()} speakers, {len(test_df)} utterances")
print(f"speaker overlap: {set(train_df['speaker_id']) & set(test_df['speaker_id'])}")


def kaldi_normalize_text(text):
    # Lowercase + strip punctuation. Deliberately does NOT collapse u/o the
    # way main.ipynb's normalize_bisaya does for WER scoring -- that merge
    # is a scoring-time leniency, not a valid training transcript for a
    # model that's supposed to learn the u/o distinction from audio.
    text = text.lower()
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_kaldi_data_dir(df, out_dir, wav_out_dir):
    out_dir = Path(out_dir)
    wav_out_dir = Path(wav_out_dir)

    if (out_dir / "wav.scp").exists():
        print(f"[skip] {out_dir}: wav.scp already exists")
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    wav_out_dir.mkdir(parents=True, exist_ok=True)

    wav_lines, text_lines, utt2spk_lines = [], [], []

    for i, row in df.iterrows():
        # utt-ids are prefixed with speaker-id -- Kaldi's sort-order
        # conventions require this for utt2spk/spk2utt to line up.
        spk = str(row["speaker_id"])
        utt_id = f"{spk}-{i:06d}"

        wav_path = wav_out_dir / f"{utt_id}.wav"
        wav_path.write_bytes(row["audio"]["bytes"])

        # Piped through sox so every utterance ends up at one consistent
        # sample rate/channel count regardless of the source file's own rate.
        wav_lines.append(f"{utt_id} sox {wav_path} -r 16000 -c 1 -t wav - |")

        transcript = kaldi_normalize_text(str(row["transcript"]))
        text_lines.append(f"{utt_id} {transcript}")
        utt2spk_lines.append(f"{utt_id} {spk}")

    (out_dir / "wav.scp").write_text("\n".join(sorted(wav_lines)) + "\n")
    (out_dir / "text").write_text("\n".join(sorted(text_lines)) + "\n")
    (out_dir / "utt2spk").write_text("\n".join(sorted(utt2spk_lines)) + "\n")

    sh(f"utils/utt2spk_to_spk2utt.pl {out_dir}/utt2spk > {out_dir}/spk2utt", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {out_dir}", cwd=WORK_DIR)
    sh(f"utils/validate_data_dir.sh --no-feats {out_dir}", cwd=WORK_DIR)

    print(f"{out_dir}: {len(wav_lines)} utterances, {df['speaker_id'].nunique()} speakers")


build_kaldi_data_dir(train_df, DATA_ROOT / "train", WORK_DIR / "wav" / "train")
build_kaldi_data_dir(test_df, DATA_ROOT / "test", WORK_DIR / "wav" / "test")
save_checkpoint()

## 9. Speed Perturbation (Training Data Only)

Kaldi's standard 3-way speed perturbation (0.9x/1.0x/1.1x), applied only to
the training set -- matches `train.ipynb` Section 7. Skipped if
`train_sp/wav.scp` already exists.

In [ ]:
TRAIN_DATA_DIR = DATA_ROOT / "train_sp"

def _speed_perturb():
    sh(f"utils/data/perturb_data_dir_speed_3way.sh {DATA_ROOT}/train {TRAIN_DATA_DIR}", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {TRAIN_DATA_DIR}", cwd=WORK_DIR)

stage(
    "speed perturbation",
    lambda: (TRAIN_DATA_DIR / "wav.scp").exists(),
    _speed_perturb,
)

## 10. Lexicon and Phoneme Data (`data/local/dict`, PS27)

Builds a lexicon constrained to **PS27**'s 27-monophone inventory, exactly
as in `train.ipynb` Section 8 -- see that notebook for the full rationale.
Lexicon build and `prepare_lang.sh` are each skipped if their own output
already exists.

In [ ]:
PS27_PHONES = {
    "p", "b", "t", "d", "k", "g",
    "f", "v", "s", "z", "sh", "th", "h",
    "j", "ch",
    "m", "n", "ng",
    "l", "r",
    "w", "y",
    "a", "e", "i", "o", "u",
}


def word_to_phones(word):
    w = re.sub(r"[^a-z']", "", word.lower()).replace("'", "")
    phones = []
    i = 0
    while i < len(w):
        if w[i:i + 2] == "ng":
            phones.append("ng")
            i += 2
        elif w[i:i + 2] == "sh":
            phones.append("sh")
            i += 2
        elif w[i:i + 2] == "th":
            phones.append("th")
            i += 2
        elif w[i:i + 2] in ("ts", "ty"):
            phones.append("ch")
            i += 2
        elif w[i] == "c":
            phones.append("s" if w[i + 1:i + 2] in ("e", "i") else "k")
            i += 1
        elif w[i] == "q":
            phones.append("k")
            i += 1
        elif w[i] == "x":
            phones.extend(["k", "s"])
            i += 1
        else:
            phones.append(w[i])
            i += 1

    assert all(p in PS27_PHONES for p in phones), f"{word} -> {phones} outside PS27"
    return phones


def build_lexicon(text_paths, dict_dir):
    dict_dir = Path(dict_dir)

    if (dict_dir / "lexicon.txt").exists():
        print(f"[skip] {dict_dir}: lexicon.txt already exists")
        return dict_dir

    dict_dir.mkdir(parents=True, exist_ok=True)

    vocab = set()
    for p in text_paths:
        for line in Path(p).read_text().splitlines():
            words = line.split(" ")[1:]
            vocab.update(w for w in words if w)

    lexicon_lines = ["<unk> spn"]
    all_phones = set()
    for word in sorted(vocab):
        phones = word_to_phones(word)
        if not phones:
            continue
        all_phones.update(phones)
        lexicon_lines.append(f"{word} {' '.join(phones)}")

    (dict_dir / "lexicon.txt").write_text("\n".join(lexicon_lines) + "\n")
    (dict_dir / "silence_phones.txt").write_text("sil\nspn\n")
    (dict_dir / "optional_silence.txt").write_text("sil\n")
    (dict_dir / "nonsilence_phones.txt").write_text("\n".join(sorted(all_phones)) + "\n")
    (dict_dir / "extra_questions.txt").write_text("")

    print(f"Vocabulary: {len(vocab)} words, {len(all_phones)} distinct phones (of PS27's 27) -> {dict_dir}")
    return dict_dir


LOCAL_DICT_DIR = build_lexicon(
    [DATA_ROOT / "train" / "text", DATA_ROOT / "test" / "text"],
    DATA_ROOT / "local" / "dict",
)

LANG_DIR = DATA_ROOT / "lang"

def _prepare_lang():
    sh(f"utils/prepare_lang.sh --position-dependent-phones false "
       f"{LOCAL_DICT_DIR} '<unk>' {DATA_ROOT}/local/lang {LANG_DIR}", cwd=WORK_DIR)

stage("prepare_lang", lambda: (LANG_DIR / "L.fst").exists(), _prepare_lang)

## 11. Language Model (KenLM -> ARPA -> Kaldi `G.fst`)

Builds a **2-gram** word-level LM from the training transcripts -- matches
`train.ipynb` Section 9 (`conclusion.md` reports 2-gram as best for Bisaya).
Skipped if `lang_2g/G.fst` already exists.

In [ ]:
LM_DIR = DATA_ROOT / "local" / "lm"
LANG_2G_DIR = DATA_ROOT / "lang_2g"

def _build_lm():
    kenlm_bin = KALDI_ROOT / "tools" / "kenlm" / "build" / "bin"
    LM_DIR.mkdir(parents=True, exist_ok=True)

    train_text_lines = (DATA_ROOT / "train" / "text").read_text().splitlines()
    corpus_txt = LM_DIR / "corpus.txt"
    corpus_txt.write_text("\n".join(" ".join(line.split(" ")[1:]) for line in train_text_lines) + "\n")

    arpa_path = LM_DIR / "2gram.arpa"
    sh(f"{kenlm_bin}/lmplz -o 2 --discount_fallback < {corpus_txt} > {arpa_path}")
    sh(f"utils/format_lm.sh {LANG_DIR} {arpa_path} {LOCAL_DICT_DIR}/lexicon.txt "
       f"{LANG_2G_DIR}", cwd=WORK_DIR)

stage("build 2-gram LM", lambda: (LANG_2G_DIR / "G.fst").exists(), _build_lm)

print("lang_2g graph directory:", LANG_2G_DIR)

## 12. MFCC + CMVN Feature Extraction

25 ms window, 10 ms frameshift, 13 static coefficients -- Kaldi's MFCC
defaults already match this. CMVN is per-speaker. Matches `train.ipynb`
Section 10. Skipped per split if that split's `feats.scp` already exists
and is non-empty.

In [ ]:
N_JOBS = min(8, os.cpu_count() or 4)

def _make_extract_fn(name, data_dir):
    def _fn():
        sh(f"steps/make_mfcc.sh --cmd run.pl --nj {N_JOBS} {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
        sh(f"steps/compute_cmvn_stats.sh {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
        sh(f"utils/fix_data_dir.sh {data_dir}", cwd=WORK_DIR)
    return _fn


for name, data_dir in [("train", TRAIN_DATA_DIR), ("test", DATA_ROOT / "test")]:
    feats_scp = Path(data_dir) / "feats.scp"
    stage(
        f"MFCC+CMVN ({name})",
        lambda p=feats_scp: p.exists() and p.stat().st_size > 0,
        _make_extract_fn(name, data_dir),
        checkpoint=False,
    )

save_checkpoint()

## 13. HMM-GMM Training (Monophone -> Triphone -> LDA+MLLT -> SAT)

The standard Kaldi progression: monophone, then triphone (delta features),
then LDA+MLLT, then speaker-adaptive training (SAT) -- SAT is this
notebook's **final model**, matching `train.ipynb` Section 11. Each stage is
skipped if its own `final.mdl` already exists.

In [ ]:
def _train_mono():
    sh(f"steps/train_mono.sh --cmd run.pl --nj {N_JOBS} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono {EXP_ROOT}/mono_ali", cwd=WORK_DIR)

stage("monophone (mono)", lambda: (EXP_ROOT / "mono" / "final.mdl").exists(), _train_mono)


def _train_tri1():
    sh(f"steps/train_deltas.sh --cmd run.pl 2000 10000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono_ali {EXP_ROOT}/tri1", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri1 {EXP_ROOT}/tri1_ali", cwd=WORK_DIR)

stage("triphone (tri1, delta features)", lambda: (EXP_ROOT / "tri1" / "final.mdl").exists(), _train_tri1)


def _train_tri2():
    sh(f"steps/train_lda_mllt.sh --cmd run.pl 2500 15000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri1_ali {EXP_ROOT}/tri2", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri2 {EXP_ROOT}/tri2_ali", cwd=WORK_DIR)

stage("LDA+MLLT (tri2)", lambda: (EXP_ROOT / "tri2" / "final.mdl").exists(), _train_tri2)


def _train_tri3():
    sh(f"steps/train_sat.sh --cmd run.pl 2500 15000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri2_ali {EXP_ROOT}/tri3", cwd=WORK_DIR)

stage("SAT (tri3) -- final model", lambda: (EXP_ROOT / "tri3" / "final.mdl").exists(), _train_tri3)

## 14. Decode + Evaluate the Final Model (SAT, 2-gram, PS27)

Builds the decode graph from `lang_2g` and the `tri3` SAT model, decodes the
test set, and sweeps the LM weight over 1-25 before printing the best WER --
matches `train.ipynb` Section 12. Compare the printed number against
`conclusion.md`'s reported 5.41% WER for this exact configuration (expect a
gap -- this is a different, smaller corpus).

Graph building and decoding are each skipped if already done; the final
WER printout always re-runs (cheap) even when decoding itself was skipped,
so you always see the result.

In [ ]:
GRAPH_DIR = EXP_ROOT / "tri3" / "graph"
DECODE_DIR = EXP_ROOT / "tri3" / "decode_test"

def _mkgraph():
    sh(f"utils/mkgraph.sh {LANG_2G_DIR} {EXP_ROOT}/tri3 {GRAPH_DIR}", cwd=WORK_DIR)

stage("build decode graph (mkgraph)", lambda: (GRAPH_DIR / "HCLG.fst").exists(), _mkgraph)


def _decode():
    sh(f"steps/decode_fmllr.sh --cmd run.pl --nj {N_JOBS} "
       f"--min-lmwt 1 --max-lmwt 25 "
       f"{GRAPH_DIR} {DATA_ROOT}/test {DECODE_DIR}", cwd=WORK_DIR)

stage(
    "decode test set",
    lambda: DECODE_DIR.exists() and any(DECODE_DIR.glob("wer_*")),
    _decode,
)

# Cheap to recompute/print every time, even when decoding itself was skipped.
sh(f"grep WER {DECODE_DIR}/wer_* | utils/best_wer.sh", cwd=WORK_DIR)

## 15. Summary

The Section 14 best-WER line is this notebook's final result: a 2-gram,
3-state, SAT HMM-GMM model using PS27 -- the single configuration
`conclusion.md` identifies as the thesis's best-performing Bisaya model
overall (5.41% WER on their corpus). Compare your printed number against
that figure, expecting a gap due to corpus differences (vocabulary size,
domain, speaker count, total duration) -- see `train.ipynb`'s Summary for
the full caveats, particularly around Section 10's rule-based G2P lexicon
being an approximation of the paper's own transcriber-produced PS27
dictionary.

If a Kaggle session ends mid-run, just restart the session, re-attach the
checkpoint dataset, call `restore_checkpoint()`, and re-run this notebook
top to bottom -- every stage above will skip everything already completed
and pick up exactly where it left off.